<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/week7_day2_dalychallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Challenge Quotidien : Le Reranking avec Pinecone Serverless
Ce notebook vous guide à travers l'installation et l'utilisation d'un modèle de 'Rerank' pour améliorer la précision des recherches sémantiques.

In [1]:
# Étape 1 : Installation des bibliothèques Pinecone nécessaires
# Nous utilisons la version 6.0.1 comme recommandé
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 10.8 MB/s eta 0:00:00


In [3]:
import os
# Étape 2 : Authentification avec Pinecone
# Si la clé API n'est pas déjà présente, Colab affichera un champ pour la saisir
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

# Étape 3 : Initialisation du client Pinecone
from pinecone import Pinecone

# On récupère la clé après l'authentification pour s'assurer qu'elle est présente
api_key = os.environ.get("PINECONE_API_KEY")

if not api_key:
    print("Erreur : Clé API introuvable. Veuillez exécuter la cellule et saisir votre clé.")
else:
    pc = Pinecone(api_key=api_key)               # Création de l'instance du client
    print("Client Pinecone initialisé avec succès.")

Client Pinecone initialisé avec succès.


In [4]:
# Étape 4 : Définition de la requête et des documents de test
# Le but est de tester si le modèle distingue Apple (l'entreprise) de la pomme (le fruit)
query = "Parle-moi des produits Apple"

documents = [
    "La pomme Granny Smith est un fruit vert et acide.", # Document sur le fruit
    "L'iPhone 15 Pro est le dernier smartphone d'Apple.", # Document sur l'entreprise
    "Les vergers de pommiers nécessitent beaucoup de soleil.", # Fruit
    "Le MacBook Air M3 est un ordinateur portable puissant.", # Entreprise
    "Steve Jobs a cofondé Apple dans un garage en Californie."	# Entreprise / Histoire
]

In [5]:
# Étape 5 : Appel du modèle de Reranking
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3", # Le modèle utilisé pour réévaluer la pertinence
    query=query,                # Notre question
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)], # Transformation en format compatible
    top_n=3                     # Nous voulons les 3 meilleurs résultats
)

# Étape 6 : Inspection des résultats réorganisés
def show_reranked_results(query, rerank_output):
    print(f"Requête : {query}\n")
    # On accède aux résultats via l'attribut '.data'
    for i, m in enumerate(rerank_output.data):
        print(f"Rang {i+1} : Score = {m.score:.4f}")
        print(f"Texte : {m.document.text}\n")

# Appel de la fonction pour afficher les résultats
show_reranked_results(query, reranked)

Requête : Parle-moi des produits Apple

Rang 1 : Score = 0.0242
Texte : Le MacBook Air M3 est un ordinateur portable puissant.

Rang 2 : Score = 0.0220
Texte : L'iPhone 15 Pro est le dernier smartphone d'Apple.

Rang 3 : Score = 0.0004
Texte : Steve Jobs a cofondé Apple dans un garage en Californie.



## Partie 2 : Configuration d'un Index Serverless pour des Notes Médicales

In [6]:
# Installation des outils pour la manipulation de données et les modèles de langage
!pip install pandas torch transformers

In [7]:
import time
import pandas as pd
from pinecone import ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Configuration de l'environnement Pinecone
cloud = 'aws'           # Fournisseur cloud par défaut
region = 'us-east-1'    # Région par défaut
index_name = 'medical-notes-index' # Nom de votre index

# Spécification pour un index sans serveur (Serverless)
spec = ServerlessSpec(cloud=cloud, region=region)

# Nettoyage : suppression de l'index s'il existe déjà pour recommencer à zéro
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Création du nouvel index
pc.create_index(
    name=index_name,
    dimension=384,     # Doit correspondre à la taille du modèle 'all-MiniLM-L6-v2'
    metric='cosine',    # La mesure de similarité recommandée pour le texte
    spec=spec
)

{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-9cuqwjj.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [13]:
import requests
import tempfile
import pandas as pd

# Téléchargement des données de test (notes médicales)
# Utilisation de l'URL brute directe pour éviter les erreurs de redirection ou de dossier
url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl"

try:
    response = requests.get(url)
    response.raise_for_status()

    # Chargement dans un DataFrame Pandas
    with tempfile.NamedTemporaryFile(delete=False, suffix='.jsonl') as tmp:
        tmp.write(response.content)
        tmp_path = tmp.name

    df = pd.read_json(tmp_path, orient='records', lines=True)

    # Aperçu des données
    print("Dimensions du tableau :", df.shape)
    display(df.head())
except Exception as e:
    print(f"Erreur lors du téléchargement : {e}")

Dimensions du tableau : (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [14]:
# Connexion à l'index et envoi des données (Upsert)
index = pc.Index(name=index_name)

# Envoi des données du DataFrame vers l'index Pinecone
index.upsert_from_dataframe(df)

# Attente que les données soient indexées et prêtes
def is_fresh(index_obj):
    stats = index_obj.describe_index_stats()
    count = stats.total_vector_count
    print(f"Nombre de vecteurs indexés : {count}")
    return count > 0 # On attend d'avoir au moins un vecteur

while not is_fresh(index):
    time.sleep(5) # Pause de 5 secondes avant de revérifier

print("L'index est prêt !")

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

Nombre de vecteurs indexés : 100
L'index est prêt !


In [15]:
# Définition de la fonction d'encodage (Embedding)
def get_embedding(text):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    inputs = tokenizer(text, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
        # Moyenne sur la dimension 1 (les tokens) pour obtenir un vecteur unique
        embedding = outputs.last_hidden_state[0].mean(dim=0)
    return embedding.tolist()

# Test de recherche sémantique
question = "Le patient a-t-il des antécédents de douleurs thoraciques ?"
query_vector = get_embedding(question)

# Recherche des 5 résultats les plus proches
results = index.query(vector=[query_vector], top_k=5, include_metadata=True)

# Affichage des résultats bruts
print(f"Question : {question}")
for i, match in enumerate(results['matches']):
    print(f"{i+1}. ID: {match['id']} | Score: {match['score']:.4f}")
    print(f"   Note: {match['metadata']}\n")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Question : Le patient a-t-il des antécédents de douleurs thoraciques ?
1. ID: P001 | Score: 0.3429
   Note: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

2. ID: P047 | Score: 0.3313
   Note: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

3. ID: P095 | Score: 0.3313
   Note: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

4. ID: P079 | Score: 0.3255
   Note: {'recovery': 'well', 'surgery': 'appendix removal'}

5. ID: P092 | Score: 0.3222
   Note: {'condition': 'dehydration', 'treatment': 'IV fluids'}



In [16]:
# Reranking final des notes médicales
refined_query = "Recherche spécifique sur les symptômes de cardiopathie ou douleur au thorax"

# Préparation des documents pour le reranker
transformed_docs = [
    {
        'id': m['id'],
        'text': '; '.join([f"{k}: {v}" for k, v in m['metadata'].items()])
    }
    for m in results['matches']
]

# Exécution du Rerank sur les résultats de la recherche initiale
final_rerank = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_docs,
    top_n=2,
    return_documents=True
)

# Affichage des résultats finaux
print("--- Résultats après Reranking ---")
for i, res in enumerate(final_rerank.data):
    print(f"{i+1}. ID: {res.document.id} | Nouveau Score: {res.score:.4f}")
    print(f"   Contenu : {res.document.text[:200]}...\n")

--- Résultats après Reranking ---
1. ID: P001 | Nouveau Score: 0.0938
   Contenu : symptoms: chest pain; tests: EKG, stress test...

2. ID: P047 | Nouveau Score: 0.0044
   Contenu : symptoms: back pain; treatment: physical therapy...

